# DementiaSense: Stages 7-12
## Feature Engineering, Baseline, Modelling, Tuning, Evaluation, Use Case & Limitations

**Prerequisite:** run the Stages 2-6 notebook first. It creates `outputs/train.csv`, `outputs/test.csv` and `outputs/feature_config.json`, which this notebook loads.

**Rules of the game (they come straight from the guidelines):**
- The **test set is used once**, in Stage 11, to report results. It is **never** used to choose features, models, hyperparameters or thresholds. Those choices use cross-validation on the training set.
- Cells marked *(write this yourself)* are for your justification. Write them from your actual outputs.
- Set the two decision variables in the config cell (`BALANCING`, and later `USE_ENGINEERED` / `SELECT_K`) from **your own** Stage 5 and Stage 7 conclusions.
- **Run time:** Stage 10 fits many models. With the default settings expect roughly 10-30 minutes on a laptop. Set `FAST_MODE = True` for a quick smoke test (results are then not meaningful).

In [ ]:
import warnings, random, platform, json, functools
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from scipy import stats
from scipy.stats import loguniform

import sklearn
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.calibration import calibration_curve
from sklearn.exceptions import UndefinedMetricWarning
from sklearn.metrics import (average_precision_score, roc_auc_score, roc_curve, precision_recall_curve,
                             confusion_matrix, f1_score, balanced_accuracy_score, accuracy_score,
                             brier_score_loss)

try:
    from IPython.display import display
except ImportError:
    display = print

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UndefinedMetricWarning)
warnings.filterwarnings("ignore", category=UserWarning)

try:
    from imblearn.pipeline import Pipeline as ImbPipeline
    from imblearn.over_sampling import SMOTE
    from imblearn.under_sampling import RandomUnderSampler
    HAS_IMBLEARN = True
except ImportError:
    HAS_IMBLEARN = False

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 170)

In [ ]:
# =========================== CONFIG: edit these from YOUR earlier conclusions ===========================
OUT_DIR = Path("outputs")
FIG_DIR = OUT_DIR / "figures"; FIG_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR = OUT_DIR / "models"; MODEL_DIR.mkdir(parents=True, exist_ok=True)

BALANCING = "class_weight"   # your Stage 5 decision: "none" | "class_weight" | "smote" | "undersample"
RECALL_TARGET = 0.90         # screening goal: catch at least this share of true cases (chosen on training CV, not on test)
PRIMARY_SET = "A_screening"  # the feature set treated as the main screening model in Stage 12 demos
TUNE_SETS = ["A_screening", "B_plus_symptoms", "C_plus_cognitive_tests"]

FAST_MODE = False            # True = tiny search / few bootstraps, for smoke-testing only
N_ITER = 3 if FAST_MODE else 15      # random-search candidates per model
N_BOOT = 100 if FAST_MODE else 1000  # bootstrap resamples for test-set confidence intervals
# ==========================================================================================================

if BALANCING in ("smote", "undersample") and not HAS_IMBLEARN:
    raise ImportError("BALANCING needs imbalanced-learn: pip install imbalanced-learn")

def savefig(name):
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# ---------- Load the Stage 4 hand-off ----------
needed = [OUT_DIR / "train.csv", OUT_DIR / "test.csv", OUT_DIR / "feature_config.json"]
if not all(p.exists() for p in needed):
    raise FileNotFoundError("Run the Stages 2-6 notebook first (it creates train.csv, test.csv, feature_config.json).")

cfg = json.load(open(OUT_DIR / "feature_config.json"))
SEED = cfg["seed"]; TARGET = cfg["target"]
CONTINUOUS, BINARY, ORDINAL, NOMINAL = cfg["continuous"], cfg["binary"], cfg["ordinal"], cfg["nominal"]
FEATURE_SETS = cfg["feature_sets"]
ALL_FEATURES = CONTINUOUS + BINARY + ORDINAL + NOMINAL

random.seed(SEED); np.random.seed(SEED)

train = pd.read_csv(OUT_DIR / "train.csv"); test = pd.read_csv(OUT_DIR / "test.csv")
X_train, y_train = train.drop(columns=TARGET), train[TARGET].astype(int)
X_test, y_test = test.drop(columns=TARGET), test[TARGET].astype(int)

print("Python", platform.python_version(), "| pandas", pd.__version__, "| scikit-learn", sklearn.__version__, "| seed", SEED)
print(f"Train {X_train.shape} (positive rate {y_train.mean():.3f}) | Test {X_test.shape} (positive rate {y_test.mean():.3f})")
print("Balancing strategy carried over from Stage 5:", BALANCING)

SYMPTOMS = [c for c in FEATURE_SETS["B_plus_symptoms"] if c not in FEATURE_SETS["A_screening"]]
COG_TESTS = [c for c in FEATURE_SETS["C_plus_cognitive_tests"] if c not in FEATURE_SETS["B_plus_symptoms"]]
MODES = {"A_screening": "Screening (demographics, lifestyle, history, clinical measurements)",
         "B_plus_symptoms": "Screening + reported symptoms",
         "C_plus_cognitive_tests": "Decision support (adds MMSE, functional assessment, ADL)"}
print("Symptom features:", SYMPTOMS)
print("Cognitive/functional test features:", COG_TESTS)

---
# Stage 7: Feature Engineering / Selection

**Is this stage applicable?** Yes. The raw data already has many features, so the aim is (a) a few clinically motivated derived features, and (b) a check on whether selecting fewer features helps. Both are judged by cross-validation on the training set, not assumed to help.

**Engineered features and why** (all are row-wise calculations with no learned parameters, so they cannot leak information from the test set):

| Feature | Definition | Clinical motivation |
|---|---|---|
| `PulsePressure` | Systolic - Diastolic BP | Arterial stiffness marker, linked to vascular contributions to cognitive decline |
| `Chol_HDL_Ratio` | Total / HDL cholesterol | Standard cardiovascular risk ratio |
| `LDL_HDL_Ratio` | LDL / HDL cholesterol | Atherogenic balance |
| `NonHDL_Chol` | Total - HDL cholesterol | Captures all atherogenic cholesterol |
| `CardiometabolicCount` | Cardiovascular disease + diabetes + hypertension | Cumulative vascular risk burden |
| `SymptomCount` | Number of reported symptoms (sets B and C) | Combines weak individual symptom flags into one stronger signal |

In [ ]:
CARDIO_COLS = ["CardiovascularDisease", "Diabetes", "Hypertension"]
ENG_CLINICAL = ["PulsePressure", "Chol_HDL_Ratio", "LDL_HDL_Ratio", "NonHDL_Chol", "CardiometabolicCount"]
ENG_SYMPTOM = ["SymptomCount"]
ENG_COLS = ENG_CLINICAL + ENG_SYMPTOM
ENG_ADD = {"A_screening": ENG_CLINICAL,
           "B_plus_symptoms": ENG_CLINICAL + ENG_SYMPTOM,
           "C_plus_cognitive_tests": ENG_CLINICAL + ENG_SYMPTOM}

def add_features(d):
    d = d.copy()
    d["PulsePressure"] = d["SystolicBP"] - d["DiastolicBP"]
    hdl = d["CholesterolHDL"].replace(0, np.nan)
    d["Chol_HDL_Ratio"] = d["CholesterolTotal"] / hdl
    d["LDL_HDL_Ratio"] = d["CholesterolLDL"] / hdl
    d["NonHDL_Chol"] = d["CholesterolTotal"] - d["CholesterolHDL"]
    d["CardiometabolicCount"] = d[CARDIO_COLS].sum(axis=1, min_count=1)
    d["SymptomCount"] = d[SYMPTOMS].sum(axis=1, min_count=1)
    return d.replace([np.inf, -np.inf], np.nan)

X_train_F = add_features(X_train)   # row-wise, no fitting: safe to apply to train and test separately
X_test_F = add_features(X_test)

print("Columns before:", X_train.shape[1], "| after adding engineered features:", X_train_F.shape[1])
print("Engineered columns with missing values (train):", X_train_F[ENG_COLS].isna().sum()[lambda s: s > 0].to_dict() or "none")
display(X_train_F[ENG_COLS].describe().T.round(2))

# Does each new feature carry a signal? (train only, same effect-size idea as Stage 6)
rows = []
for c in ENG_COLS:
    a, b = X_train_F.loc[y_train == 1, c].dropna(), X_train_F.loc[y_train == 0, c].dropna()
    U, p = stats.mannwhitneyu(a, b, alternative="two-sided")
    rows.append({"feature": c, "median_diagnosed": a.median(), "median_not": b.median(),
                 "rank_biserial": 2 * U / (len(a) * len(b)) - 1, "p_value": p})
display(pd.DataFrame(rows).round(4))

In [ ]:
# ---------- Redundancy check: engineered features vs the raw features they come from ----------
groups = {"BP": ["SystolicBP", "DiastolicBP", "PulsePressure"],
          "Cholesterol": ["CholesterolTotal", "CholesterolLDL", "CholesterolHDL", "CholesterolTriglycerides",
                          "NonHDL_Chol", "Chol_HDL_Ratio", "LDL_HDL_Ratio"]}
for name, cols in groups.items():
    corr = X_train_F[cols].corr(method="spearman")
    Z = X_train_F[cols].fillna(X_train_F[cols].median())
    vif = pd.Series(np.diag(np.linalg.pinv(np.corrcoef(Z.values.T))), index=cols, name="VIF")
    print(f"--- {name} ---")
    display(corr.round(2))
    display(vif.round(1).to_frame().T)
print("NonHDL_Chol and PulsePressure are exact linear combinations of raw columns, so they add redundancy for linear models.")
print("Regularised logistic regression and tree models cope with this; it is still worth testing, which is what 7.3 does.")

In [ ]:
# ---------- Feature relevance ranking: mutual information (train only) ----------
CONT_ALL = CONTINUOUS + ENG_COLS

def build_preprocessor(features):
    cont = [c for c in CONT_ALL if c in features]
    binr = [c for c in BINARY if c in features]
    ordn = [c for c in ORDINAL if c in features]
    nomn = [c for c in NOMINAL if c in features]
    parts = []
    if cont:
        parts.append(("cont", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), cont))
    if binr:
        parts.append(("bin", SimpleImputer(strategy="most_frequent"), binr))
    if ordn:
        parts.append(("ord", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("sc", StandardScaler())]), ordn))
    if nomn:
        parts.append(("nom", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                                       ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), nomn))
    return ColumnTransformer(parts, remainder="drop", verbose_feature_names_out=False)

full_feats = FEATURE_SETS["C_plus_cognitive_tests"] + ENG_COLS
pp = build_preprocessor(full_feats)
Xt = pp.fit_transform(X_train_F[full_feats])
mi = pd.Series(mutual_info_classif(Xt, y_train, random_state=SEED), index=pp.get_feature_names_out()).sort_values(ascending=False)
display(mi.head(15).round(4).to_frame("mutual_information"))

fig, ax = plt.subplots(figsize=(7, 6))
mi.head(20)[::-1].plot.barh(ax=ax, color="#5b8fb9"); ax.set_xlabel("Mutual information with Diagnosis (train)")
savefig("07_mutual_information")
print("Features with ~zero mutual information are candidates for removal; 7.3 tests whether removal helps.")

### 7.3 Do the engineered features and feature selection actually help?
Each variant is evaluated by 5-fold stratified CV on the **training set**. Selection (`SelectKBest` with mutual information) is placed **inside** the pipeline, so it is re-fitted on each training fold and cannot see the validation fold.

| Variant | Meaning |
|---|---|
| `base` | original features of the set |
| `base+engineered` | plus the engineered features |
| `base+eng+top15` | plus engineered features, then keep the 15 highest mutual-information features |

In [ ]:
MODEL_NAMES = ["LogReg", "SVM_RBF", "RandomForest", "HistGB"]
SIMPLICITY = ["LogReg", "SVM_RBF", "RandomForest", "HistGB"]   # simplest first (used for tie-breaking in Stage 11)
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
SCORING = {"pr_auc": "average_precision", "roc_auc": "roc_auc", "recall": "recall",
           "precision": "precision", "f1": "f1", "balanced_acc": "balanced_accuracy"}
MI_SCORE = functools.partial(mutual_info_classif, random_state=SEED)
SELECT_K = None       # set after Stage 7.3 (None = use all features)

def make_model(name, class_weight=None):
    if name == "LogReg":
        return LogisticRegression(max_iter=3000, class_weight=class_weight, random_state=SEED)
    if name == "SVM_RBF":
        return SVC(kernel="rbf", probability=True, class_weight=class_weight, random_state=SEED)
    if name == "RandomForest":
        return RandomForestClassifier(n_estimators=300, class_weight=class_weight, random_state=SEED)
    if name == "HistGB":
        return HistGradientBoostingClassifier(class_weight=class_weight, random_state=SEED)
    raise ValueError(name)

def make_pipeline(features, model_name, strategy=None, select_k="global"):
    strategy = BALANCING if strategy is None else strategy
    k = SELECT_K if select_k == "global" else select_k
    steps = [("prep", build_preprocessor(features))]
    if k:
        steps.append(("select", SelectKBest(MI_SCORE, k=k)))
    if strategy == "smote":
        steps.append(("smote", SMOTE(random_state=SEED)))
    elif strategy == "undersample":
        steps.append(("under", RandomUnderSampler(random_state=SEED)))
    steps.append(("clf", make_model(model_name, "balanced" if strategy == "class_weight" else None)))
    return (ImbPipeline if strategy in ("smote", "undersample") else Pipeline)(steps)

def run_cv(pipe, X, y):
    r = cross_validate(pipe, X, y, cv=CV, scoring=SCORING, return_train_score=True, n_jobs=-1)
    out = {}
    for m in SCORING:
        out[m] = r[f"test_{m}"].mean(); out[m + "_sd"] = r[f"test_{m}"].std()
    out["train_pr_auc"] = r["train_pr_auc"].mean()
    return out

def show_cv(df, id_cols):
    out = df[id_cols].copy()
    for m in SCORING:
        out[m] = df[m].round(3).astype(str) + " +/- " + df[m + "_sd"].round(3).astype(str)
    out["train_pr_auc"] = df["train_pr_auc"].round(3)
    out["train-val gap"] = (df["train_pr_auc"] - df["pr_auc"]).round(3)
    return out

rows = []
for set_name, base in FEATURE_SETS.items():
    variants = {"base": (base, None),
                "base+engineered": (base + ENG_ADD[set_name], None),
                "base+eng+top15": (base + ENG_ADD[set_name], 15)}
    for model_name in ["LogReg", "RandomForest"]:
        for vname, (feats, k) in variants.items():
            pipe = make_pipeline(feats, model_name, select_k=k)
            rows.append({"feature_set": set_name, "model": model_name, "variant": vname, **run_cv(pipe, X_train_F[feats], y_train)})
exp7 = pd.DataFrame(rows)
display(show_cv(exp7, ["feature_set", "model", "variant"]))
exp7.to_csv(OUT_DIR / "stage7_feature_experiment.csv", index=False)

In [ ]:
# ---------- Decision aid ----------
piv = exp7.pivot_table(index=["feature_set", "model"], columns="variant", values="pr_auc")
sd = exp7[exp7.variant == "base"].set_index(["feature_set", "model"])["pr_auc_sd"]
gain_eng = (piv["base+engineered"] - piv["base"])
gain_sel = (piv["base+eng+top15"] - piv["base+engineered"])
display(pd.DataFrame({"PR-AUC gain: engineered vs base": gain_eng, "PR-AUC gain: top15 vs all": gain_sel, "fold SD (base)": sd}).round(4))

SUGGEST_ENG = bool(gain_eng.mean() > sd.mean())
SUGGEST_K = 15 if bool(gain_sel.mean() > sd.mean()) else None
print(f"Mean gain from engineered features: {gain_eng.mean():+.4f} (typical fold SD {sd.mean():.4f}) -> suggest keeping them: {SUGGEST_ENG}")
print(f"Mean gain from top-15 selection:    {gain_sel.mean():+.4f} -> suggest using selection: {SUGGEST_K is not None}")
print("Rule used: adopt a change only if the average gain exceeds the typical fold-to-fold spread. Override below if you can justify it.")

In [ ]:
# ---------- YOUR decision (defaults follow the rule above; override if you can justify it) ----------
USE_ENGINEERED = SUGGEST_ENG
SELECT_K = SUGGEST_K

FINAL_SETS = {name: base + (ENG_ADD[name] if USE_ENGINEERED else []) for name, base in FEATURE_SETS.items()}
print("USE_ENGINEERED =", USE_ENGINEERED, "| SELECT_K =", SELECT_K)
for k, v in FINAL_SETS.items():
    print(f"  {k}: {len(v)} features entering the pipeline")
json.dump({"use_engineered": USE_ENGINEERED, "select_k": SELECT_K, "balancing": BALANCING, "final_sets": FINAL_SETS},
          open(OUT_DIR / "stage7_decisions.json", "w"), indent=2)

### Stage 7: What / Why / How / Result  *(write this yourself)*
- **What:** which features you created or removed.
- **Why:** the clinical reasoning (use the table above, but explain it in your own words) and what the mutual-information ranking showed.
- **How:** stateless row-wise features; selection inside the CV pipeline so it cannot leak.
- **What changed because of it:** quote the PR-AUC gain with its spread. If the gain is inside the spread, say the features did not help and why you kept or dropped them.
- **Validated?** by 5-fold CV on the training set only.

---
# Stage 8: Baseline Model

**Purpose:** a reference point that any "advanced" model must beat. Four baselines, all using the **original features only**, with no tuning and no engineered features:

| Baseline | Role |
|---|---|
| `Dummy_prior` | Always predicts the majority class. Shows what "accuracy without skill" looks like |
| `Dummy_stratified` | Random guesses with the training class frequencies |
| `DecisionTree_depth3` | Tiny, fully interpretable rule set |
| `LogReg_default` | **Main baseline**: a plain logistic regression with default settings and no balancing |

In [ ]:
def baseline_pipe(features, name):
    prep = build_preprocessor(features)
    if name == "LogReg_default":
        return make_pipeline(features, "LogReg", strategy="none", select_k=None)
    clf = {"Dummy_prior": DummyClassifier(strategy="prior"),
           "Dummy_stratified": DummyClassifier(strategy="stratified", random_state=SEED),
           "DecisionTree_depth3": DecisionTreeClassifier(max_depth=3, random_state=SEED)}[name]
    return Pipeline([("prep", prep), ("clf", clf)])

rows = []
for set_name, feats in FEATURE_SETS.items():
    for b in ["Dummy_prior", "Dummy_stratified", "DecisionTree_depth3", "LogReg_default"]:
        rows.append({"feature_set": set_name, "baseline": b, **run_cv(baseline_pipe(feats, b), X_train_F[feats], y_train)})
exp8 = pd.DataFrame(rows)
display(show_cv(exp8, ["feature_set", "baseline"]))
exp8.to_csv(OUT_DIR / "stage8_baseline_cv.csv", index=False)

print("Positive-class prevalence in train (= PR-AUC of a model with no skill):", round(y_train.mean(), 3))

### Stage 8: What / Why / How / Result  *(write this yourself)*
- **How to read the table:** a dummy model's PR-AUC equals the prevalence and its ROC-AUC is about 0.5. Any real model must clear these. Note recall and F1 for `Dummy_prior`: high accuracy would be possible with zero recall, which is why accuracy is not used.
- **Result:** the PR-AUC / recall / F1 of `LogReg_default` for each feature set. This is the number the final models must beat **by more than the fold spread**.
- Discuss how much the baseline improves from set A to B to C. That jump is your evidence about how much the closer-to-diagnosis features contribute.

---
# Stage 9: ML / Advanced ML Modelling

**Models compared and why**

| Model | Key assumptions | Why it suits (or not) this problem |
|---|---|---|
| Logistic regression (L2) | Roughly linear effect on the log-odds; features scaled; independent rows | Interpretable (coefficients, odds ratios), stable on ~2k rows, the natural step up from the baseline |
| SVM (RBF kernel) | Smooth non-linear boundary; features scaled; sensitive to C and gamma | Captures non-linear structure; probability outputs come from internal cross-validation so it is slower |
| Random forest | No linearity or scaling assumption; many deep trees averaged | Handles interactions and outliers; robust default for tabular data; can overfit noisy features |
| Histogram gradient boosting | Sequentially corrected shallow trees | Often the strongest tabular learner; more hyperparameters and a higher overfitting risk on small data |

**Deep learning: not applicable.** With roughly 1.7k rows of structured tabular features, a neural network is unlikely to beat boosted trees, is harder to explain to clinicians, and the guidelines allow DL only "where justified". That is the justification for excluding it.

All four use the balancing choice from Stage 5 (`BALANCING`) and the features from Stage 7. Training scores are reported next to validation scores to expose overfitting. A higher training score is **not** treated as improvement.

In [ ]:
rows = []
for set_name in TUNE_SETS:
    feats = FINAL_SETS[set_name]
    for m in MODEL_NAMES:
        rows.append({"feature_set": set_name, "model": m, **run_cv(make_pipeline(feats, m), X_train_F[feats], y_train)})
exp9 = pd.DataFrame(rows)
display(show_cv(exp9, ["feature_set", "model"]))
exp9.to_csv(OUT_DIR / "stage9_model_comparison_cv.csv", index=False)

# Compare against the Stage 8 baseline
base_ref = exp8[exp8.baseline == "LogReg_default"].set_index("feature_set")["pr_auc"]
cmp9 = exp9.assign(baseline_pr_auc=exp9.feature_set.map(base_ref))
cmp9["gain_vs_baseline"] = cmp9["pr_auc"] - cmp9["baseline_pr_auc"]
display(cmp9[["feature_set", "model", "pr_auc", "pr_auc_sd", "baseline_pr_auc", "gain_vs_baseline"]].round(3))

In [ ]:
piv9 = exp9.pivot(index="model", columns="feature_set", values="pr_auc")[TUNE_SETS]
piv9.plot.bar(figsize=(8, 4), rot=0); plt.ylabel("CV PR-AUC (train folds)")
plt.axhline(y_train.mean(), color="grey", ls="--", lw=1, label="no-skill (prevalence)")
plt.legend(fontsize=8); plt.title("Model comparison before tuning")
savefig("09_model_comparison")

### Stage 9: What / Why / How / Result  *(write this yourself)*
- Which models beat the baseline by more than their fold spread, and which do not?
- Look at the train-vs-validation gap: a large gap means the model memorises the training data.
- Does the ranking of models change between feature sets?
- If all models perform about the same, say so: it suggests the features, not the algorithm, are the limit, and the simplest model is the defensible choice.

---
# Stage 10: Hyperparameter Tuning

**Method:** randomised search (`RandomizedSearchCV`) with 5-fold stratified CV on the training set. Random search covers wide ranges more efficiently than a full grid, especially for the continuous parameters (C, gamma, learning rate).

**Selection metric:** PR-AUC (`average_precision`). It is threshold-independent and focuses on the minority (diagnosed) class. Recall, F1 and ROC-AUC are recorded as well. The decision threshold is chosen separately in Stage 11.

**Caution:** the best CV score from a search is slightly optimistic, because it is the maximum over many candidates. That is why the held-out test set in Stage 11 is the real check.

In [ ]:
PARAM_DISTS = {
    "LogReg": {"clf__C": loguniform(1e-3, 1e2)},
    "SVM_RBF": {"clf__C": loguniform(1e-2, 1e2), "clf__gamma": loguniform(1e-4, 1e0)},
    "RandomForest": {"clf__n_estimators": [100, 200, 300], "clf__max_depth": [None, 4, 6, 8, 12],
                     "clf__min_samples_leaf": [1, 2, 5, 10, 20], "clf__max_features": ["sqrt", "log2", 0.5]},
    "HistGB": {"clf__learning_rate": loguniform(0.01, 0.3), "clf__max_leaf_nodes": [8, 15, 31],
               "clf__min_samples_leaf": [10, 20, 40], "clf__l2_regularization": loguniform(1e-3, 10),
               "clf__max_iter": [100, 200, 300]},
}

tuned = {}
for set_name in TUNE_SETS:
    feats = FINAL_SETS[set_name]
    for m in MODEL_NAMES:
        s = RandomizedSearchCV(make_pipeline(feats, m), PARAM_DISTS[m], n_iter=N_ITER, scoring=SCORING,
                               refit="pr_auc", cv=CV, n_jobs=-1, random_state=SEED)
        s.fit(X_train_F[feats], y_train)
        tuned[(set_name, m)] = s
        print(f"done: {set_name:24s} {m:13s} best CV PR-AUC = {s.best_score_:.3f}")

In [ ]:
def clean(v):
    return v.item() if hasattr(v, "item") else v

rows, params_out = [], {}
for (set_name, m), s in tuned.items():
    i = s.best_index_
    d = exp9[(exp9.feature_set == set_name) & (exp9.model == m)].iloc[0]
    best_params = {k.replace("clf__", ""): clean(v) for k, v in s.best_params_.items()}
    params_out[f"{set_name}|{m}"] = best_params
    rows.append({"feature_set": set_name, "model": m,
                 "default CV PR-AUC": round(d.pr_auc, 3),
                 "tuned CV PR-AUC": round(s.cv_results_["mean_test_pr_auc"][i], 3),
                 "tuned SD": round(s.cv_results_["std_test_pr_auc"][i], 3),
                 "gain": round(s.cv_results_["mean_test_pr_auc"][i] - d.pr_auc, 3),
                 "tuned recall": round(s.cv_results_["mean_test_recall"][i], 3),
                 "tuned F1": round(s.cv_results_["mean_test_f1"][i], 3),
                 "best params": best_params})
tune_tbl = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 120)
display(tune_tbl)
tune_tbl.drop(columns="best params").to_csv(OUT_DIR / "stage10_tuning_summary.csv", index=False)
json.dump(params_out, open(OUT_DIR / "stage10_best_params.json", "w"), indent=2, default=str)

### Stage 10: What / Why / How / Result  *(write this yourself)*
- Which hyperparameters mattered (look at `cv_results_`, or compare the best parameters with the defaults) and why that makes sense (for example, stronger regularisation on small noisy data).
- How much did tuning improve things? If `gain` is inside the SD, tuning changed nothing reliable. That is an honest result, not a failure.
- Mention the ranges you searched and why (the `PARAM_DISTS` cell).

---
# Stage 11: Validation & Evaluation

**Order of work (no peeking at the test set before step 4):**
1. Choose the final model per feature set from **CV results** (one-standard-error rule: among models within one SD of the best PR-AUC, take the simplest).
2. Obtain out-of-fold (OOF) probabilities on the training set and choose the **decision threshold** so that recall >= `RECALL_TARGET`.
3. Refit on the full training set (already done by the search).
4. **Evaluate once on the untouched test set**, next to the baseline, with bootstrap confidence intervals.
5. Error analysis and explainability.

**Metrics and why:** the task is imbalanced classification where a missed case costs more than a false alarm, so the headline metrics are **PR-AUC, recall, precision, F1, balanced accuracy and ROC-AUC**, plus the confusion matrix. Accuracy alone is not used. The Brier score and calibration curve check whether the predicted probabilities can be read as risks.

In [ ]:
FINAL_MODEL_OVERRIDE = {}   # e.g. {"A_screening": "LogReg"}  (only with a written justification)

def pick_final(set_name):
    rows = []
    for m in MODEL_NAMES:
        s = tuned[(set_name, m)]; i = s.best_index_
        rows.append({"model": m, "cv_pr_auc": s.cv_results_["mean_test_pr_auc"][i], "cv_sd": s.cv_results_["std_test_pr_auc"][i]})
    t = pd.DataFrame(rows)
    best = t.loc[t.cv_pr_auc.idxmax()]
    eligible = t[t.cv_pr_auc >= best.cv_pr_auc - best.cv_sd]
    choice = sorted(eligible.model, key=SIMPLICITY.index)[0]
    return FINAL_MODEL_OVERRIDE.get(set_name, choice), t, best.model

def choose_threshold(y, p, target):
    prec, rec, thr_ = precision_recall_curve(y, p)
    ok = np.where(rec[:-1] >= target)[0]
    return float(thr_[ok.max()]) if len(ok) else float(thr_[0])

def metrics_at(y, p, thr):
    y = np.asarray(y); p = np.asarray(p)
    pred = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {"pr_auc": average_precision_score(y, p), "roc_auc": roc_auc_score(y, p), "brier": brier_score_loss(y, p),
            "threshold": thr, "recall": tp / (tp + fn) if tp + fn else np.nan,
            "precision": tp / (tp + fp) if tp + fp else np.nan, "specificity": tn / (tn + fp) if tn + fp else np.nan,
            "f1": f1_score(y, pred, zero_division=0), "balanced_acc": balanced_accuracy_score(y, pred),
            "accuracy": accuracy_score(y, pred), "TP": tp, "FP": fp, "TN": tn, "FN": fn}

chosen, final_models, thr, oof, test_p, base_models, base_p = {}, {}, {}, {}, {}, {}, {}
sel_rows = []
for set_name in TUNE_SETS:
    name, t, best_name = pick_final(set_name)
    chosen[set_name] = name
    feats = FINAL_SETS[set_name]
    est = tuned[(set_name, name)].best_estimator_
    final_models[set_name] = est
    oof[set_name] = cross_val_predict(clone(est), X_train_F[feats], y_train, cv=CV, method="predict_proba", n_jobs=-1)[:, 1]
    thr[set_name] = choose_threshold(y_train, oof[set_name], RECALL_TARGET)
    bf = FEATURE_SETS[set_name]
    base_models[set_name] = baseline_pipe(bf, "LogReg_default").fit(X_train_F[bf], y_train)
    # ---- test set touched here for the first time: predictions only, nothing is selected from them ----
    test_p[set_name] = est.predict_proba(X_test_F[feats])[:, 1]
    base_p[set_name] = base_models[set_name].predict_proba(X_test_F[bf])[:, 1]
    sel_rows.append({"feature_set": set_name, "best CV PR-AUC model": best_name, "chosen (1-SE rule)": name,
                     "threshold (OOF, recall>=%.2f)" % RECALL_TARGET: round(thr[set_name], 3)})
display(pd.DataFrame(sel_rows))

In [ ]:
# ---------- 11.1 Training vs cross-validation vs test (final model) ----------
cols = ["pr_auc", "roc_auc", "recall", "precision", "f1", "balanced_acc", "specificity"]
rows = []
for set_name in TUNE_SETS:
    feats = FINAL_SETS[set_name]; t_ = thr[set_name]
    ptrain = final_models[set_name].predict_proba(X_train_F[feats])[:, 1]
    for label, y_, p_ in [("train (in-sample)", y_train, ptrain), ("CV out-of-fold", y_train, oof[set_name]),
                          ("TEST (held out)", y_test, test_p[set_name])]:
        m = metrics_at(y_, p_, t_)
        rows.append({"feature_set": set_name, "model": chosen[set_name], "data": label, **{c: round(m[c], 3) for c in cols}})
perf = pd.DataFrame(rows)
display(perf)
perf.to_csv(OUT_DIR / "stage11_train_cv_test.csv", index=False)
print("A large drop from train to CV/test indicates overfitting. Only the TEST rows count as evidence of performance on unseen data.")

In [ ]:
# ---------- 11.2 Final vs baseline on the test set ----------
rows = []
for set_name in TUNE_SETS:
    m_final = metrics_at(y_test, test_p[set_name], thr[set_name])
    m_base05 = metrics_at(y_test, base_p[set_name], 0.5)
    rows.append({"feature_set": set_name, "model": f"baseline LogReg (thr 0.5)", **{c: round(m_base05[c], 3) for c in cols}})
    rows.append({"feature_set": set_name, "model": f"final {chosen[set_name]} (thr {thr[set_name]:.2f})", **{c: round(m_final[c], 3) for c in cols}})
cmp11 = pd.DataFrame(rows)
display(cmp11)
cmp11.to_csv(OUT_DIR / "stage11_final_vs_baseline_test.csv", index=False)

# Confusion counts at the chosen threshold
cm_rows = []
for set_name in TUNE_SETS:
    m = metrics_at(y_test, test_p[set_name], thr[set_name])
    cm_rows.append({"feature_set": set_name, "TP": m["TP"], "FN": m["FN"], "FP": m["FP"], "TN": m["TN"]})
display(pd.DataFrame(cm_rows))

In [ ]:
# ---------- 11.3 Bootstrap 95% confidence intervals on the test set ----------
KEYS = ["pr_auc", "roc_auc", "recall", "precision", "f1", "balanced_acc"]

def bootstrap(y, p_final, p_base, t_final, n=N_BOOT, seed=SEED):
    rng = np.random.default_rng(seed)
    y = np.asarray(y); pf = np.asarray(p_final); pb = np.asarray(p_base)
    vals, diffs = [], []
    for _ in range(n):
        b = rng.integers(0, len(y), len(y))
        if y[b].min() == y[b].max():
            continue
        m = metrics_at(y[b], pf[b], t_final)
        vals.append([m[k] for k in KEYS])
        diffs.append(average_precision_score(y[b], pf[b]) - average_precision_score(y[b], pb[b]))
    return np.array(vals), np.array(diffs)

ci_rows, diff_rows = [], []
for set_name in TUNE_SETS:
    vals, diffs = bootstrap(y_test, test_p[set_name], base_p[set_name], thr[set_name])
    pt = metrics_at(y_test, test_p[set_name], thr[set_name])
    lo, hi = np.nanpercentile(vals, [2.5, 97.5], axis=0)
    row = {"feature_set": set_name, "model": chosen[set_name]}
    for k, l, h in zip(KEYS, lo, hi):
        row[k] = f"{pt[k]:.3f} [{l:.3f}, {h:.3f}]"
    ci_rows.append(row)
    dl, dh = np.percentile(diffs, [2.5, 97.5])
    diff_rows.append({"feature_set": set_name, "PR-AUC gain vs baseline (point)":
                      round(average_precision_score(y_test, test_p[set_name]) - average_precision_score(y_test, base_p[set_name]), 3),
                      "95% CI": f"[{dl:.3f}, {dh:.3f}]", "share of resamples > 0": round((diffs > 0).mean(), 3)})
display(pd.DataFrame(ci_rows))
display(pd.DataFrame(diff_rows))
pd.DataFrame(ci_rows).to_csv(OUT_DIR / "stage11_test_bootstrap_ci.csv", index=False)
print("If the CI of the gain over the baseline includes 0, do not claim the final model is better.")

In [ ]:
# ---------- 11.4 Curves, threshold, confusion matrices, calibration ----------
n = len(TUNE_SETS)
fig, axes = plt.subplots(2, n, figsize=(5 * n, 8.5), squeeze=False)
for j, s in enumerate(TUNE_SETS):
    for p_, lab, ls in [(test_p[s], f"final {chosen[s]}", "-"), (base_p[s], "baseline LogReg", "--")]:
        fpr, tpr, _ = roc_curve(y_test, p_)
        axes[0, j].plot(fpr, tpr, ls, label=f"{lab} (AUC {roc_auc_score(y_test, p_):.2f})")
        pr, rc, _ = precision_recall_curve(y_test, p_)
        axes[1, j].plot(rc, pr, ls, label=f"{lab} (AP {average_precision_score(y_test, p_):.2f})")
    axes[0, j].plot([0, 1], [0, 1], color="grey", lw=0.8); axes[0, j].set_title(f"ROC: {s}"); axes[0, j].set_xlabel("FPR"); axes[0, j].set_ylabel("TPR")
    axes[1, j].axhline(y_test.mean(), color="grey", lw=0.8); axes[1, j].set_title(f"Precision-Recall: {s}"); axes[1, j].set_xlabel("Recall"); axes[1, j].set_ylabel("Precision")
    axes[0, j].legend(fontsize=8); axes[1, j].legend(fontsize=8)
savefig("11_roc_pr_test")

fig, axes = plt.subplots(1, n, figsize=(4.6 * n, 4), squeeze=False)
for j, s in enumerate(TUNE_SETS):
    m = metrics_at(y_test, test_p[s], thr[s])
    sns.heatmap([[m["TN"], m["FP"]], [m["FN"], m["TP"]]], annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[0, j],
                xticklabels=["Pred no", "Pred yes"], yticklabels=["True no", "True yes"])
    axes[0, j].set_title(f"{s}\n{chosen[s]}, threshold {thr[s]:.2f}")
savefig("11_confusion_matrices")

fig, axes = plt.subplots(1, n, figsize=(4.6 * n, 4), squeeze=False)
for j, s in enumerate(TUNE_SETS):
    prec, rec, th = precision_recall_curve(y_train, oof[s])
    axes[0, j].plot(th, rec[:-1], label="recall"); axes[0, j].plot(th, prec[:-1], label="precision")
    axes[0, j].axvline(thr[s], color="red", ls="--", lw=1, label=f"chosen {thr[s]:.2f}")
    axes[0, j].axhline(RECALL_TARGET, color="grey", ls=":", lw=1)
    axes[0, j].set_title(f"Threshold choice (train OOF): {s}"); axes[0, j].set_xlabel("threshold"); axes[0, j].legend(fontsize=8)
savefig("11_threshold_choice")

fig, axes = plt.subplots(1, n, figsize=(4.6 * n, 4), squeeze=False)
for j, s in enumerate(TUNE_SETS):
    fr, mp = calibration_curve(y_test, test_p[s], n_bins=8, strategy="quantile")
    axes[0, j].plot(mp, fr, marker="o", label=f"Brier {brier_score_loss(y_test, test_p[s]):.3f}"); axes[0, j].plot([0, 1], [0, 1], color="grey", lw=0.8)
    axes[0, j].set_title(f"Calibration (test): {s}"); axes[0, j].set_xlabel("mean predicted probability"); axes[0, j].set_ylabel("observed fraction"); axes[0, j].legend(fontsize=8)
savefig("11_calibration")

In [ ]:
# ---------- 11.5 Explainability: permutation importance on the test set (explanation only, no selection) ----------
perm = {}
for s in TUNE_SETS:
    feats = FINAL_SETS[s]
    r = permutation_importance(final_models[s], X_test_F[feats], y_test, scoring="average_precision",
                               n_repeats=5 if FAST_MODE else 20, random_state=SEED, n_jobs=-1)
    perm[s] = pd.DataFrame({"feature": feats, "importance": r.importances_mean, "sd": r.importances_std}).sort_values("importance", ascending=False)

fig, axes = plt.subplots(1, n, figsize=(5.2 * n, 5), squeeze=False)
for j, s in enumerate(TUNE_SETS):
    d = perm[s].head(12).iloc[::-1]
    axes[0, j].barh(d.feature, d.importance, xerr=d.sd, color="#5b8fb9")
    axes[0, j].set_title(f"{s}\n({chosen[s]})", fontsize=10); axes[0, j].set_xlabel("drop in PR-AUC when shuffled")
savefig("11_permutation_importance")
display(perm[PRIMARY_SET].head(10).round(4))

# Logistic regression: readable coefficients (standardised features -> comparable magnitudes)
for s in TUNE_SETS:
    if chosen[s] == "LogReg":
        est = final_models[s]
        names = est.named_steps["prep"].get_feature_names_out()
        if "select" in est.named_steps:
            names = names[est.named_steps["select"].get_support()]
        coef = pd.Series(est.named_steps["clf"].coef_[0], index=names).sort_values(key=abs, ascending=False)
        print(f"LogReg coefficients, {s} (log-odds per 1 SD / per unit for 0-1 features), top 10:")
        display(coef.head(10).round(3).to_frame("coef"))

In [ ]:
# ---------- 11.6 Error analysis (primary screening model) ----------
pm = PRIMARY_SET
te = X_test_F.copy()
te["y"] = y_test.values; te["p"] = test_p[pm]; te["pred"] = (te["p"] >= thr[pm]).astype(int)
te["outcome"] = np.select([(te.y == 1) & (te.pred == 1), (te.y == 1) & (te.pred == 0), (te.y == 0) & (te.pred == 1)],
                          ["TP", "FN", "FP"], "TN")
print(f"Error analysis for {pm} ({chosen[pm]}), threshold {thr[pm]:.3f}")
display(te.outcome.value_counts().to_frame("count").T)

top_feats = perm[pm].feature.head(6).tolist()
display(te.groupby("outcome")[top_feats].mean().T.round(2))

print("Most confidently MISSED cases (false negatives, lowest predicted risk):")
display(te[te.outcome == "FN"].sort_values("p").head(8)[["p"] + top_feats].round(3))
print("Most confident FALSE ALARMS (false positives, highest predicted risk):")
display(te[te.outcome == "FP"].sort_values("p", ascending=False).head(8)[["p"] + top_feats].round(3))

In [ ]:
# ---------- 11.7 Save final models and metrics ----------
for s in TUNE_SETS:
    joblib.dump({"model": final_models[s], "features": FINAL_SETS[s], "threshold": thr[s], "mode": s,
                 "algorithm": chosen[s], "recall_target": RECALL_TARGET, "sklearn_version": sklearn.__version__, "seed": SEED},
                MODEL_DIR / f"final_{s}.joblib")
json.dump({s: {"algorithm": chosen[s], "threshold": thr[s]} for s in TUNE_SETS}, open(OUT_DIR / "thresholds.json", "w"), indent=2)
print("Saved:", sorted(p.name for p in MODEL_DIR.glob("*.joblib")))

### Stage 11: What / Why / How / Result  *(write this yourself)*
Answer each of these in your own words, using only your real outputs:
1. **Which model/feature set is best, and how was that decided?** (CV, 1-SE rule, not the test set.)
2. **Is it better than the baseline?** Quote the test PR-AUC gain and its bootstrap CI. If the CI includes 0, the honest answer is "not demonstrably".
3. **Train vs CV vs test:** any overfitting?
4. **Threshold:** what recall did you target, why, and what precision (false-alarm cost) did that imply? Show the trade-off.
5. **Calibration:** can the probability be read as a risk?
6. **Errors:** who gets missed (false negatives) and why? What do the false alarms have in common?
7. **Explainability:** which features drive the predictions, and does it make clinical sense? If features you expected to matter (age, family history, BMI) have no influence, discuss what that says about the dataset.
8. **Unseen data:** how would performance change on new patients? (See Stage 12 robustness checks.)
9. **What changed from A to B to C?** That comparison is the answer to your leakage / early-screening question.

---
# Stage 12: Use Case, Impact & Limitations

This stage has two parts: evidence (robustness, subgroup checks, an inference demo) and your written analysis.

In [ ]:
# ---------- 12.1 Robustness to incomplete inputs (simulated missing values in the TEST data) ----------
rng = np.random.default_rng(SEED)
rows = []
for s in TUNE_SETS:
    feats = FINAL_SETS[s]
    for frac in [0.0, 0.1, 0.2, 0.3]:
        Xm = X_test[ALL_FEATURES].copy()
        if frac > 0:
            Xm = Xm.mask(rng.random(Xm.shape) < frac)
        p_ = final_models[s].predict_proba(add_features(Xm)[feats])[:, 1]
        m = metrics_at(y_test, p_, thr[s])
        rows.append({"feature_set": s, "missing %": int(frac * 100), "pr_auc": round(m["pr_auc"], 3),
                     "roc_auc": round(m["roc_auc"], 3), "recall": round(m["recall"], 3), "precision": round(m["precision"], 3)})
rob = pd.DataFrame(rows)
display(rob)
rob.to_csv(OUT_DIR / "stage12_missing_data_robustness.csv", index=False)
print("Values are blanked at random and imputed by the pipeline (median / most frequent from the TRAINING data).")

In [ ]:
# ---------- 12.2 Subgroup performance (primary screening model) ----------
sg = X_test_F[["Gender", "Ethnicity", "EducationLevel", "Age"]].copy()
sg["AgeGroup"] = pd.cut(sg["Age"], bins=[0, 69, 79, 200], labels=["<70", "70-79", "80+"])
sg["y"] = y_test.values; sg["p"] = test_p[PRIMARY_SET]

rows = []
for col in ["Gender", "Ethnicity", "EducationLevel", "AgeGroup"]:
    for level, g in sg.groupby(col, observed=True):
        n_pos = int(g.y.sum())
        enough = len(g) >= 30 and 5 <= n_pos <= len(g) - 5
        m = metrics_at(g.y, g.p, thr[PRIMARY_SET]) if enough else None
        rows.append({"variable": col, "level": level, "n": len(g), "positives": n_pos,
                     "recall": round(m["recall"], 3) if m else np.nan,
                     "precision": round(m["precision"], 3) if m else np.nan,
                     "pr_auc": round(m["pr_auc"], 3) if m else np.nan})
subg = pd.DataFrame(rows)
display(subg)
subg.to_csv(OUT_DIR / "stage12_subgroup_performance.csv", index=False)
print("Subgroups are small, so differences are noisy. Use this table to flag where to look, not to prove fairness or bias.")

In [ ]:
# ---------- 12.3 Inference demo: how the saved model would score a NEW patient ----------
def predict_risk(patient, mode=PRIMARY_SET):
    art = joblib.load(MODEL_DIR / f"final_{mode}.joblib")
    row = pd.DataFrame([patient]).reindex(columns=ALL_FEATURES).apply(pd.to_numeric, errors="coerce")
    row = add_features(row)                               # missing inputs become NaN and are imputed by the pipeline
    p = float(art["model"].predict_proba(row[art["features"]])[:, 1][0])
    flagged = p >= art["threshold"]
    return {"mode": mode, "model": art["algorithm"], "risk_probability": round(p, 3), "threshold": round(art["threshold"], 3),
            "recommendation": ("REFER: higher dementia risk, suggest specialist cognitive assessment" if flagged
                               else "Lower estimated risk: routine monitoring"),
            "note": "Screening aid only. Not a diagnosis. A clinician makes the decision."}

demo = X_test[ALL_FEATURES].iloc[:3]
for i in range(3):
    print("Actual label (for reference):", int(y_test.iloc[i]), "->", predict_risk(demo.iloc[i].to_dict()))
print()
print("Same patient with several inputs missing:")
partial = demo.iloc[0].to_dict()
for k in list(partial)[:12]:
    partial.pop(k)
print(predict_risk(partial))

### 12.4 Use case, impact and limitations  *(write this yourself)*

Use the evidence above plus your own reasoning. Suggested structure:

| Question | What to cover |
|---|---|
| **Who benefits?** | GPs and community health workers (triage which patients to refer), memory clinics (better-prioritised referrals), patients and families (earlier assessment) |
| **How is it used?** | Inputs a clinician can collect cheaply go in; a risk probability and a refer / monitor flag come out. Screening mode (set A/B) versus decision-support mode (set C) |
| **Practical impact** | What your test numbers imply per 100 patients screened at your chosen threshold: how many true cases caught, how many false alarms. Compute it from your own confusion matrix, do not guess |
| **Risks** | False negatives give false reassurance; false positives cause anxiety and unnecessary referrals; over-reliance on the tool; privacy of health data |
| **Limitations to state honestly** | Dataset provenance (real or synthetic, from Stage 2); one dataset and no external validation; small test set (see the CI widths); population and geography unknown; subgroup sizes too small to establish fairness; features such as MMSE are close to the diagnosis; correlation is not causation; no prospective or clinical validation |
| **Future work** | External validation on a real cohort (e.g. OASIS-2, with a patient-level split), calibration tuning, adding blood biomarkers or MRI measures, a web or API interface and usability testing with clinicians, monitoring for data drift |

**Do not claim** deployment, clinical validity or real-world impact that you did not demonstrate. The guidelines treat such claims as misconduct.

### Stage 12: What / Why / How / Result  *(write this yourself)*

---
# Reproducibility and Academic-Integrity Record

In [ ]:
print("Seed:", SEED, "| Python", platform.python_version(), "| scikit-learn", sklearn.__version__,
      "| pandas", pd.__version__, "| numpy", np.__version__)
print("imbalanced-learn available:", HAS_IMBLEARN)
print()
print("Decisions taken in this notebook:")
print("  Balancing:", BALANCING, "| engineered features used:", USE_ENGINEERED, "| select_k:", SELECT_K)
print("  Final models:", chosen)
print("  Thresholds (recall target %.2f):" % RECALL_TARGET, {k: round(v, 3) for k, v in thr.items()})
print()
print("Files written to", OUT_DIR.resolve())
for p in sorted(OUT_DIR.rglob("*")):
    if p.is_file() and p.suffix in (".csv", ".json", ".joblib", ".png"):
        print(" ", p.relative_to(OUT_DIR))

**Record of resources (fill in for your report):**

| Resource | Details |
|---|---|
| Dataset | name, URL, author, licence, date accessed |
| Libraries | pandas, numpy, scikit-learn, scipy, matplotlib, seaborn, joblib, imbalanced-learn (versions printed above) |
| Code sources / AI assistance | list any tool used and what you verified yourself |
| References | cite only papers you have actually read |